In [8]:
!sudo apt-get update
!sudo apt-get install python3.11 python3.11-distutils


Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading packag

In [9]:
!curl -sS https://bootstrap.pypa.io/get-pip.py | sudo python3.11


  Using cached pip-25.3-py3-none-any.whl.metadata (4.7 kB)
Using cached pip-25.3-py3-none-any.whl (1.8 MB)
  Attempting uninstall: pip
    Found existing installation: pip 25.3
    Uninstalling pip-25.3:
      Successfully uninstalled pip-25.3


In [20]:
!python3.11 -m pip install ipykernel
!python3.11 -m pip uninstall -y transformers  # Uninstall any global transformers to prevent conflicts
!python3.11 -m ipykernel install --name py311 --display-name "Python 3.11"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 23.4 MB/s eta 0:00:00
Found existing installation: transformers 4.53.2
Uninstalling transformers-4.53.2:
  Successfully uninstalled transformers-4.53.2
0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
Installed kernelspec py311 in /usr/local/share/jupyter/kernels/py311


In [16]:
import sys
sys.version


'3.11.13 (main, Jun  4 2025, 08:57:29) [GCC 11.4.0]'

In [2]:

!uv run pytest test/ir/graph/test_create_masegraph.py
!uv run python -c "import chop; import transformers; print('Environment ready!')"

Test session starts (platform: linux, Python 3.11.9, pytest 9.0.2, pytest-sugar 1.1.1)
rootdir: /content/mase
configfile: pytest.ini
plugins: xdist-3.8.0, html-4.1.1, sugar-1.1.1, metadata-3.1.1, cov-7.0.0, profiling-1.8.1
collected 235 items                                                            

 test/ir/graph/test_create_masegraph.py ✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓ 11% █▏        
                                        ✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓ 21% ██▎       
                                        ✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓ 32% ███▎      
                                        ✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓ 43% ████▍     
                                        ✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓ 53% █████▍    
                                        ✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓ 64% ██████▍   
                                        ✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓ 74% ███████▌  
                                        ✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓✓ 85% ████████▌ 
                                        ✓✓✓✓✓✓✓✓

## Implementation Details :

def graph_iterator_quantize_by_type(graph, config: dict):
    # Some modules might need information from two graphs to be initialized
    if (
        config.get("baseline_weight_path") is not None
        and config.get("load_type") == "mz"
    ):
        bl_graph = deepcopy_mase_graph(graph)
        bl_graph = load_mase_graph_interface_pass(
            bl_graph, pass_args=config.get("baseline_weight_path")
        )
    else:
        bl_graph = None
    for node in graph.fx_graph.nodes:
        if node.meta["mase"]["common"].get("mase_op", None) is None:
            logger.debug(
                f"Skipping node: {node.name} because mase op was not found. This may be a serialization issue with checkpoint export/load."
            )
            continue
        if get_mase_op(node) not in QUANTIZEABLE_OP:
            continue
        node_config = get_config(config, get_mase_op(node))
        if node_config["name"] is None:
            continue
        node_config = parse_node_config(node_config, get_mase_op(node))
        # if get_mase_type(node) == "module":
        if node.op == "call_module":
            ori_module = get_node_actual_target(node)
            successor_module = get_similar_node_actual_target(
                bl_graph, node.next
            )  # Certain modules require information about their successor module
            bl_module = get_similar_node_actual_target(bl_graph, node)
            new_module = create_new_module(
                get_mase_op(node),
                ori_module,
                node_config,
                node.meta,
                bl_module,
                successor_module,
            )
            parent_name, name = get_parent_name(node.target)
            setattr(graph.modules[parent_name], name, new_module)
            # update precision and type in meta.parameters["common"]
            update_quant_meta_param(node, node_config, get_mase_op(node))
        elif get_mase_type(node) in [
            "builtin_func",
            "module_related_func",
        ]:
            new_f, args, kwargs = create_new_fn(node, node_config)
            with graph.fx_graph.inserting_before(node):
                new_node = graph.fx_graph.call_function(new_f, args, kwargs)
                new_node.name = node.name
                new_node.meta["mase"] = copy(node.meta["mase"])
                # new_node.meta["mase"].node -> new_node
                relink_node_meta(new_node, model=graph.model)
                update_quant_meta_param(new_node, node_config, get_mase_op(node))
                node.replace_all_uses_with(new_node)
            graph.fx_graph.erase_node(node)
    return graph

def update_quant_meta_param(node, config: dict, mase_op: str) -> None:
    quant_arith = config["name"]
    assert quant_arith in quant_arith_to_list_fn, f"Unknown quant_arith: {quant_arith}"
    """
    MASE_OP_TO_INPUT_ENTRIES_AND_ARGS: Give a mapping between config file and mase model
    How it works:
        We find the precision of a certain paramter "e.g data_in" using the precision partial function.

        The precision partial function take a config file and entry "e.g data_in",
        and it will search through all the attributes under this entry based on the quantisation scheme,
        returning a list of precision with the order same as attributes defined in QUANT_ARITH_TO_SUFFIXES

        This precision list is then being mapped to mase data using 'arg'
    """
    for entry, arg in zip(*MASE_OP_TO_INPUT_ENTRIES_AND_ARGS[mase_op]):
        if not arg_exists(node, arg):
            continue
        update_arg(
            node,
            arg_name=arg,
            dtype=quant_arith,
            precision=quant_arith_to_list_fn[quant_arith](config, entry),
        )

    for entry, arg in zip(*MASE_OP_TO_OUTPUT_ENTRIES[mase_op]):
        # Quantise all the output to fixed point
        if quant_arith == "binary" or quant_arith == "binary_residual":
            update_result(
                node,
                output_name=arg,
                dtype="binary",
                precision=[32, 0, 1],  # [bitwidth, stochastic, bipolar]
            )

def update_arg(node, arg_name, dtype=None, precision=None, size=None):
    if dtype is not None:
        node.meta["mase"].parameters["common"]["args"][arg_name]["type"] = dtype
    if precision is not None:
        node.meta["mase"].parameters["common"]["args"][arg_name][
            "precision"
        ] = precision
    if size is not None:
        node.meta["mase"].parameters["common"]["args"][arg_name]["size"] = size

def update_result(node, output_name, dtype=None, precision=None, size=None):
    if dtype is not None:
        node.meta["mase"].parameters["common"]["results"][output_name]["type"] = dtype
    if precision is not None:
        node.meta["mase"].parameters["common"]["results"][output_name][
            "precision"
        ] = precision
    if size is not None:
        node.meta["mase"].parameters["common"]["results"][output_name]["size"] = size




In [1]:
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]
!uv --version
!git clone https://github.com/DeepWok/mase.git
%cd mase
!uv sync
import glob, site, sys

venv_site = glob.glob("/content/mase/.venv/lib/python*/site-packages")[0]
site.addsitedir(venv_site)
sys.path.insert(0, venv_site)

print("Using:", venv_site)

downloading uv 0.9.27 x86_64-unknown-linux-gnu
no checksums to verify
installing to /usr/local/bin
  uv
  uvx
everything's installed!
uv 0.9.27
Cloning into 'mase'...
remote: Enumerating objects: 30343, done.
remote: Counting objects: 100% (1385/1385), done.
remote: Compressing objects: 100% (420/420), done.
remote: Total 30343 (delta 1148), reused 1029 (delta 949), pack-reused 28958 (from 3)
Receiving objects: 100% (30343/30343), 241.63 MiB | 17.33 MiB/s, done.
Resolving deltas: 100% (19113/19113), done.
/content/mase
Using CPython 3.11.9
Creating virtual environment at: .venv
Resolved 242 packages in 1ms
Prepared 239 packages in 55.17s
Installed 239 packages in 1.93s
 + absl-py==2.3.1
 + accelerate==1.12.0
 + accessible-pygments==0.0.5
 + aiohappyeyeballs==2.6.1
 + aiohttp==3.13.2
 + aiosignal==1.4.0
 + alabaster==1.0.0
 + ale-py==0.11.2
 + alembic==1.17.2
 + annotated-types==0.7.0
 + asttokens==3.0.1
 + attr-dot-dict==0.1.0
 + attrs==25.4.0
 + babel==2.17.0
 + beautifulsoup4==4.14.3

---

## Tasks

In [ ]:
import importlib
import src.lab1
importlib.reload(src.lab1)

from src.lab1 import *

In [ ]:
dataset, tokenizer, base_model = prep_env()
save_dir = make_save_dir("mase_bert_experiments")


Multiple distributions found for package optimum. Picked distribution: optimum
INFO     Tokenizing dataset stanfordnlp/imdb with AutoTokenizer for bert-base-uncased.
/content/mase/.venv/lib/python3.11/site-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

# Task 1 : Quantization

In [ ]:
quant_results = run_quantization_sweep(
    widths=list(range(4, 33)),
    dataset=dataset, tokenizer=tokenizer, save_dir=save_dir,
)
plot_quantization_sweep(quant_results, save_dir / "task1.png")

# Pruning

In [ ]:
quant_results = load_quantization_results(save_dir)

In [ ]:
best = get_best_result(quant_results)
prune_results = run_pruning_sweep(
    sparsities=[round(s, 1) for s in np.arange(0.1, 1.0, 0.1)],
    methods=["random", "l1-norm"],
    dataset=dataset, tokenizer=tokenizer, save_dir=save_dir,
    best_quant_config=best["quant_config"],
)
plot_pruning_sweep(prune_results, save_dir / "task2.png")